## 0. Imports

In [0]:
import time
import requests
from datetime import date, datetime, timezone
from zoneinfo import ZoneInfo

# 01_ingest_entsoe_prices — Ingestion des prix day-ahead

**Projet :** Swiss Hydro & Power Analytics
**Source :** API REST de l'ENTSO-E Transparency Platform
**Destination :** Volume `energy_raw.landing`, dossier `entsoe/prices`

Ce notebook récupère les prix day-ahead de l'électricité en Suisse et dépose la réponse XML **telle quelle** dans la zone de landing.
 Aucune transformation n'est faite ici : la lecture et la mise en table Delta se feront dans la couche Bronze.

| Étape | Description |
|---|---|
| 1 | Lecture sécurisée du token depuis un secret Databricks |
| 2 | Appel de l'API ENTSO-E |
| 3 | Enregistrement du XML brut dans le landing |


## 1. Authentification

Le token ENTSO-E est stocké dans le secret scope `entsoe` (clé `api-token`), créé avec la CLI Databricks.
Il n'apparaît jamais en clair : ni dans le code, ni dans les résultats (`[REDACTED]`), ni sur GitHub.


In [0]:
# Lecture du token ENTSO-E depuis le secret Databricks
token = dbutils.secrets.get(scope="entsoe", key="api-token")

# Arrêt immédiat si le secret est vide (sans jamais afficher sa valeur)
if not token:
    raise ValueError("Le secret entsoe/api-token est vide")

## 2. Appel de l'API

| Paramètre | Valeur | Signification |
|---|---|---|
| `documentType` | `A44` | Prix day-ahead |
| `in_Domain` / `out_Domain` | `10YCH-SWISSGRIDZ` | Zone de prix Suisse |
| `periodStart` / `periodEnd` | format `yyyyMMddHHmm` | Période demandée, **en UTC** |

⚠️ L'API travaille en UTC : minuit en heure suisse correspond à 22:00 UTC en été et 23:00 UTC en hiver.

## 2.1 Extraction mois par mois

La période demandée est découpée en tranches d'un mois maximum.
Pour chaque mois : appel de l'API, contrôle de la réponse, puis enregistrement du XML brut dans le landing.

- Une erreur sur un mois n'interrompt pas la boucle : elle est signalée dans le récapitulatif final.
- Une pause d'une seconde entre deux appels respecte les limites de débit de l'API.

In [0]:
dbutils.widgets.text("start_date", "2025-01-01", "Date de début (heure suisse)")
dbutils.widgets.text("end_date", "2026-09-01", "Date de fin, exclue (heure suisse)")
dbutils.widgets.text("area_code", "10YCH-SWISSGRIDZ", "Zone (code EIC)")
START_DATE = dbutils.widgets.get("start_date")
END_DATE = dbutils.widgets.get("end_date")
AREA_CODE = dbutils.widgets.get("area_code")
def to_entsoe_utc(date_str):
    """Convertit minuit heure suisse au format UTC de l'API : yyyyMMddHHmm."""
    local_midnight = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=ZoneInfo("Europe/Zurich"))
    return local_midnight.astimezone(ZoneInfo("UTC")).strftime("%Y%m%d%H%M")

print(START_DATE, END_DATE)

## 3. Enregistrement dans le landing

La réponse est écrite sans modification dans `/Volumes/workspace/energy_raw/landing/entsoe/prices`.

Convention de nommage : `prices_<zone>_<début>_<fin>_<horodatage d'ingestion>.xml`

Chaque exécution crée un nouveau fichier au lieu d'écraser le précédent. On conserve ainsi l'historique de ce que l'API a renvoyé, utile lorsque l'ENTSO-E publie des corrections a posteriori.


In [0]:
API_URL = "https://web-api.tp.entsoe.eu/api"
LANDING_PATH = "/Volumes/workspace/energy_raw/landing/entsoe/prices"

def month_ranges(start_str, end_str):
    """Découpe la période [début, fin[ en tranches d'un mois maximum."""
    current = datetime.strptime(start_str, "%Y-%m-%d").date()
    end = datetime.strptime(end_str, "%Y-%m-%d").date()
    while current < end:
        next_month = date(current.year + current.month // 12, current.month % 12 + 1, 1)
        chunk_end = min(next_month, end)
        yield current.isoformat(), chunk_end.isoformat()
        current = chunk_end

results = []

for chunk_start, chunk_end in month_ranges(START_DATE, END_DATE):
    params = {
        "securityToken": token,
        "documentType": "A44",                       # prix day-ahead
        "in_Domain": AREA_CODE,
        "out_Domain": AREA_CODE,
        "periodStart": to_entsoe_utc(chunk_start),   # conversion en UTC
        "periodEnd": to_entsoe_utc(chunk_end),
    }

    response = requests.get(API_URL, params=params, timeout=120)

    # Succès uniquement si code 200 ET document de données (pas un message d'erreur)
    if response.status_code == 200 and "Publication_MarketDocument" in response.text:
        ingestion_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        file_name = f"prices_CH_{params['periodStart']}_{params['periodEnd']}_{ingestion_ts}.xml"
        with open(f"{LANDING_PATH}/{file_name}", "w", encoding="utf-8") as f:
            f.write(response.text)
        status = "OK"
    else:
        status = f"ERREUR {response.status_code}"

    results.append((chunk_start, chunk_end, status))
    print(f"{chunk_start} → {chunk_end} : {status}")

    time.sleep(1)   # pause entre deux appels

# Récapitulatif
display(spark.createDataFrame(results, ["debut", "fin", "statut"]))